# Vitrine — treino do detector no SKU-110K

Este notebook treina o detector que falta ao projeto e produz **o número real** para
`benchmarks/results.md`, que hoje diz *não medido*.

## Antes de apertar play

**1. Ligue a GPU.** Menu `Ambiente de execução` → `Alterar o tipo de ambiente de execução` →
Acelerador de hardware: **T4 GPU**. Sem isso o treino leva dias em vez de horas.

**2. Nada disso roda na sua máquina.** O dataset (~13 GB) baixa para o disco do Google, o
treino usa a GPU do Google. A única coisa que desce para o seu computador é o peso final,
de aproximadamente 6 MB, na última célula.

**3. Tempo esperado.** A célula 4 faz um teste rápido de 1 época em 2% do dataset — se
algo estiver quebrado, você descobre em minutos em vez de duas horas. Só depois disso
rode o treino de verdade.

**4. A sessão gratuita cai.** Depois de ~90 min sem interação, ou ~12 h no total. Deixe a
aba aberta e volte nela de tempos em tempos.

## Licença

O **SKU-110K** é distribuído para uso acadêmico e não comercial. Este treino é para
portfólio e estudo, o que está dentro desses termos. Não redistribua o peso resultante
comercialmente.

## 1. Confirmar que a GPU está ligada

Se a saída disser `CUDA disponivel: False`, volte e ligue a T4 antes de seguir.

In [ ]:
import subprocess

print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)

import torch

print("CUDA disponivel:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise SystemExit(
        "Sem GPU. Ambiente de execucao > Alterar o tipo de ambiente de execucao > T4 GPU."
    )

## 2. Instalar o Ultralytics

In [ ]:
%pip install -q "ultralytics>=8.3"

import ultralytics

ultralytics.checks()
print("ultralytics", ultralytics.__version__)

## 3. Parâmetros do treino

Três escolhas que importam, e o motivo de cada uma:

**`MAX_DET = 1000`** — o padrão do Ultralytics é 300 detecções por imagem. O SKU-110K tem
imagens com mais de 500 produtos. Com o padrão, o recall fica **artificialmente baixo** e
você mediria o limite do parâmetro em vez da qualidade do modelo. É a pegadinha mais fácil
de cair nesse dataset.

**`IMGSZ = 640`** — produto de gôndola é objeto pequeno, então 1024 daria resultado melhor.
640 é o ponto de partida honesto: cabe numa sessão gratuita. Se sobrar tempo, repita com
1024 e registre os dois números.

**`SEED = 0` e `deterministic=True`** — o projeto inteiro se apoia em reprodutibilidade;
o treino não vai ser a exceção.

In [ ]:
MODELO_BASE = "yolov8n.pt"  # o menor. Troque por yolov8s.pt se sobrar tempo de GPU.
DATASET = "SKU-110K.yaml"  # o Ultralytics baixa e converte sozinho na primeira vez.
EPOCHS = 10
IMGSZ = 640
BATCH = 16
MAX_DET = 1000
SEED = 0

print(f"{MODELO_BASE} | {EPOCHS} epocas | imgsz {IMGSZ} | batch {BATCH} | max_det {MAX_DET}")

## 4. Teste rápido antes de gastar duas horas

Uma época sobre 2% do dataset. Serve só para provar que o download funcionou, o formato
está certo e a GPU está sendo usada. **O resultado numérico daqui não vale nada** — não
registre em lugar nenhum.

A primeira execução baixa ~13 GB e converte as anotações; isso leva um tempo mesmo com
tudo certo. Se falhar, falha aqui, e não depois de duas horas de treino.

In [ ]:
import time

from ultralytics import YOLO

inicio = time.perf_counter()
teste = YOLO(MODELO_BASE)
teste.train(
    data=DATASET,
    epochs=1,
    fraction=0.02,
    imgsz=IMGSZ,
    batch=BATCH,
    seed=SEED,
    deterministic=True,
    project="runs",
    name="sanidade",
    exist_ok=True,
    plots=False,
    val=False,
)
print(f"\nTeste de sanidade levou {(time.perf_counter() - inicio) / 60:.1f} min.")
print("Se chegou ate aqui, o caminho esta livre. Siga para a celula 5.")

## 5. O treino de verdade

**Olhe o tempo da primeira época** que aparecer no log e multiplique por `EPOCHS`. Se der
mais que umas 3 horas, interrompa, reduza `EPOCHS` e rode de novo — é melhor ter um número
real de 5 épocas que uma sessão derrubada no meio de 30.

In [ ]:
inicio = time.perf_counter()
modelo = YOLO(MODELO_BASE)
modelo.train(
    data=DATASET,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    max_det=MAX_DET,
    seed=SEED,
    deterministic=True,
    project="runs",
    name="sku110k",
    exist_ok=True,
)
duracao_min = (time.perf_counter() - inicio) / 60
print(f"\nTreino concluido em {duracao_min:.1f} min.")

## 6. Medir no split de validação

Este é o número que vale. Repare no `max_det`: sem ele, o recall sai truncado.

In [ ]:
metricas = modelo.val(
    data=DATASET,
    split="val",
    imgsz=IMGSZ,
    max_det=MAX_DET,
    plots=True,
)

precisao = float(metricas.box.mp)
recall = float(metricas.box.mr)
map50 = float(metricas.box.map50)
map5095 = float(metricas.box.map)

print(f"precisao : {precisao:.4f}")
print(f"recall   : {recall:.4f}")
print(f"mAP@50   : {map50:.4f}")
print(f"mAP@50-95: {map5095:.4f}")

## 7. Procedência: hash do peso e versões

O `ShareReport` do Vitrine carrega `weights_sha256` justamente para que dois relatórios só
sejam comparáveis quando vieram do mesmo peso. O hash sai daqui.

In [ ]:
import hashlib
from datetime import UTC, datetime
from pathlib import Path

peso = Path("runs/sku110k/weights/best.pt")
assert peso.is_file(), f"nao encontrei o peso em {peso}"

sha256 = hashlib.sha256(peso.read_bytes()).hexdigest()
tamanho_mb = peso.stat().st_size / 1_000_000
data = datetime.now(UTC).date().isoformat()

print(f"arquivo   : {peso}")
print(f"tamanho   : {tamanho_mb:.1f} MB")
print(f"sha256    : {sha256}")
print(f"data      : {data}")
print(f"ultralytics {ultralytics.__version__} | torch {torch.__version__}")

## 8. O bloco pronto para colar em `benchmarks/results.md`

Copie a saída inteira e substitua a seção **SKU-110K: NÃO MEDIDO** do arquivo.

**Cole o número que saiu, seja ele qual for.** Um mAP baixo, registrado com o método e a
procedência, vale mais que um número alto sem origem — e o projeto inteiro se apoia nisso.

In [ ]:
comando = (
    f"yolo detect train model={MODELO_BASE} data={DATASET} epochs={EPOCHS} "
    f"imgsz={IMGSZ} batch={BATCH} max_det={MAX_DET} seed={SEED} deterministic=True"
)

bloco = f"""## SKU-110K, split de validacao ({data})

| Metrica | Valor |
|---|---|
| Precisao | {precisao:.4f} |
| Recall | {recall:.4f} |
| mAP@50 | {map50:.4f} |
| mAP@50-95 | {map5095:.4f} |

**Procedencia**

- Peso: `best.pt`, {tamanho_mb:.1f} MB, sha256 `{sha256}`
- Modelo base: `{MODELO_BASE}`, treinado por {EPOCHS} epocas em {duracao_min:.0f} min
- Ambiente: Google Colab, {torch.cuda.get_device_name(0)}
- Versoes: ultralytics {ultralytics.__version__}, torch {torch.__version__}
- `max_det={MAX_DET}` na validacao. O padrao de 300 trunca o recall neste dataset,
  onde uma imagem pode ter mais de 500 produtos.

**Comando**

```bash
{comando}
```

Reproduzivel pelo notebook `notebooks/treino_sku110k.ipynb`.
"""

print(bloco)

## 9. Baixar o peso

São ~6 MB. É a única coisa deste notebook que toca o seu computador.

In [ ]:
from google.colab import files

destino = Path("vitrine_sku110k.pt")
destino.write_bytes(peso.read_bytes())
files.download(str(destino))

## 10. E agora, no Vitrine

Com o `vitrine_sku110k.pt` no seu computador:

```bash
uv pip install 'vitrine-shelf[yolo]'
uv run vitrine analyze foto.jpg --detector yolo --weights vitrine_sku110k.pt --out ./resultado
```

Isso instala o `torch` (2 a 3 GB de disco). **Se quiser evitar**, coloque o peso direto no
Hugging Face Space: a inferência roda no servidor deles e a sua máquina não muda de tamanho.

Para conferir o número na sua própria máquina, com o avaliador do projeto:

```bash
uv run vitrine benchmark ./SKU-110K --split val --detector yolo --weights vitrine_sku110k.pt --json
```

Os dois números não vão bater exatamente — o Vitrine e o Ultralytics fazem a
correspondência de caixas de formas ligeiramente diferentes, e a definição usada aqui está
escrita em `src/vitrine/eval/metrics.py`. **Se a diferença for grande, isso é um achado, não
um detalhe:** significa que uma das duas implementações está errada e vale investigar.